# X-ray 이물질 탐지 베이스라인
YOLOv3 기존 가중치(last400.pt)를 활용한 베이스라인 평가

In [ ]:
import sys, os, random
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

YOLO_DIR = '/data/knhyun/KAMP/xray_dataset/4. X-ray 검사장비 AI 데이터셋/dataset/test1/yolov3'
DATASET_DIR = '/data/knhyun/KAMP/xray_dataset/4. X-ray 검사장비 AI 데이터셋/dataset/라벨링 6종 세트'
WEIGHTS_DIR = '/data/knhyun/KAMP/xray_dataset/4. X-ray 검사장비 AI 데이터셋/dataset/실습별 가중치파일'
BASELINE_DIR = '/data/knhyun/KAMP/baseline'

sys.path.insert(0, YOLO_DIR)
from models import Darknet
from utils.utils import non_max_suppression, compute_ap, ap_per_class
import utils.torch_utils as torch_utils

device = torch_utils.select_device('')
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

## 1. 데이터셋 분할 (Train 70% / Val 15% / Test 15%)

In [ ]:
IMG_DIR = os.path.join(DATASET_DIR, 'images 400')
LABEL_DIR = os.path.join(DATASET_DIR, 'labels')

all_imgs = sorted([
    os.path.join(IMG_DIR, f) for f in os.listdir(IMG_DIR) if f.endswith('.jpg')
])

random.seed(42)
random.shuffle(all_imgs)

n = len(all_imgs)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)
train_imgs = all_imgs[:n_train]
val_imgs   = all_imgs[n_train:n_train + n_val]
test_imgs  = all_imgs[n_train + n_val:]

for name, lst in [('train', train_imgs), ('val', val_imgs), ('test', test_imgs)]:
    path = os.path.join(BASELINE_DIR, f'{name}.txt')
    with open(path, 'w') as f:
        f.write('\n'.join(lst))

print(f'전체: {n}장  |  Train: {len(train_imgs)}  |  Val: {len(val_imgs)}  |  Test: {len(test_imgs)}')

## 2. 이물질 통계 (EDA)

In [ ]:
def load_labels(img_path):
    stem = os.path.splitext(os.path.basename(img_path))[0]
    lbl_path = os.path.join(LABEL_DIR, stem + '.txt')
    if not os.path.exists(lbl_path):
        return np.zeros((0, 5))
    boxes = []
    with open(lbl_path) as f:
        for line in f:
            parts = list(map(float, line.strip().split()))
            if len(parts) == 5:
                boxes.append(parts)
    return np.array(boxes) if boxes else np.zeros((0, 5))

all_boxes = np.vstack([load_labels(p) for p in all_imgs if load_labels(p).shape[0] > 0])
widths  = all_boxes[:, 3]  # YOLO 형식: cx cy w h
heights = all_boxes[:, 4]
areas   = widths * heights
cx      = all_boxes[:, 1]
cy      = all_boxes[:, 2]

print(f'총 이물질 수: {len(all_boxes)}')
print(f'크기 (w×h) — 최소: {areas.min():.5f}, 최대: {areas.max():.5f}, 평균: {areas.mean():.5f}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(areas * 100, bins=30, color='steelblue', edgecolor='white')
axes[0].set_xlabel('이물질 면적 (% of image)', fontsize=12)
axes[0].set_ylabel('개수')
axes[0].set_title('이물질 크기 분포')

axes[1].scatter(cx, cy, s=5, alpha=0.4, color='tomato')
axes[1].set_xlim(0, 1)
axes[1].set_ylim(1, 0)
axes[1].set_xlabel('cx (normalized)')
axes[1].set_ylabel('cy (normalized)')
axes[1].set_title('이물질 위치 분포')

# 이미지별 이물질 개수
counts = [load_labels(p).shape[0] for p in all_imgs]
uniq, freq = np.unique(counts, return_counts=True)
axes[2].bar(uniq, freq, color='mediumseagreen', edgecolor='white')
axes[2].set_xlabel('이미지당 이물질 수')
axes[2].set_ylabel('이미지 수')
axes[2].set_title('이미지별 이물질 개수')
axes[2].set_xticks(uniq)

plt.tight_layout()
plt.savefig(os.path.join(BASELINE_DIR, 'eda.png'), dpi=120, bbox_inches='tight')
plt.show()
print('EDA 저장 완료')

## 3. 모델 로드 (last400.pt)

In [ ]:
CFG  = os.path.join(YOLO_DIR, 'yolov3-spp.cfg')
IMGSZ = 416

def load_model(weight_name):
    model = Darknet(CFG, IMGSZ).to(device)
    ckpt = torch.load(
        os.path.join(WEIGHTS_DIR, weight_name),
        map_location=device, weights_only=False
    )
    model.load_state_dict(ckpt['model'], strict=False)
    model.eval()
    return model

model = load_model('last400.pt')
print('last400.pt 로드 완료')

## 4. 추론 함수 정의

In [ ]:
def letterbox(img, new_shape=(416, 416), color=(114, 114, 114)):
    """이미지를 비율 유지하며 패딩 리사이즈"""
    shape = img.shape[:2]  # h, w
    r = min(new_shape[0] / shape[0], new_shape[1] / shape[1])
    new_unpad = (int(round(shape[1] * r)), int(round(shape[0] * r)))
    dw = new_shape[1] - new_unpad[0]
    dh = new_shape[0] - new_unpad[1]
    dw /= 2; dh /= 2

    import cv2
    img = cv2.resize(img, new_unpad, interpolation=cv2.INTER_LINEAR)
    top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
    left, right  = int(round(dw - 0.1)), int(round(dw + 0.1))
    img = cv2.copyMakeBorder(img, top, bottom, left, right,
                             cv2.BORDER_CONSTANT, value=color)
    return img, r, (dw, dh)

def preprocess(img_path, imgsz=416):
    import cv2
    img0 = cv2.imread(img_path)
    if img0 is None:
        img0 = np.array(Image.open(img_path).convert('RGB'))[:, :, ::-1]
    img, ratio, pad = letterbox(img0, (imgsz, imgsz))
    img = img[:, :, ::-1].transpose(2, 0, 1).copy()  # BGR→RGB, HWC→CHW
    img = torch.from_numpy(img).float().div(255.0).unsqueeze(0).to(device)
    return img, img0, ratio, pad

def xywh2xyxy(x):
    y = x.clone()
    y[:, 0] = x[:, 0] - x[:, 2] / 2
    y[:, 1] = x[:, 1] - x[:, 3] / 2
    y[:, 2] = x[:, 0] + x[:, 2] / 2
    y[:, 3] = x[:, 1] + x[:, 3] / 2
    return y

def scale_coords(img1_shape, coords, img0_shape):
    gain = min(img1_shape[0] / img0_shape[0], img1_shape[1] / img0_shape[1])
    pad  = ((img1_shape[1] - img0_shape[1] * gain) / 2,
            (img1_shape[0] - img0_shape[0] * gain) / 2)
    coords[:, [0, 2]] -= pad[0]
    coords[:, [1, 3]] -= pad[1]
    coords /= gain
    coords[:, 0].clamp_(0, img0_shape[1])
    coords[:, 1].clamp_(0, img0_shape[0])
    coords[:, 2].clamp_(0, img0_shape[1])
    coords[:, 3].clamp_(0, img0_shape[0])
    return coords

print('추론 함수 정의 완료')

## 5. 테스트셋 평가 (mAP@0.5, Precision, Recall)

In [ ]:
from utils.utils import box_iou

def evaluate(model, img_paths, conf_thres=0.001, iou_thres=0.5, imgsz=416):
    """테스트셋에 대해 mAP@0.5 계산"""
    stats = []
    model.eval()
    with torch.no_grad():
        for img_path in img_paths:
            img_t, img0, ratio, pad = preprocess(img_path, imgsz)
            h0, w0 = img0.shape[:2]

            gt = load_labels(img_path)
            if gt.shape[0] > 0:
                gt_boxes = torch.tensor(gt[:, 1:], dtype=torch.float32)
                gt_boxes_xyxy = xywh2xyxy(gt_boxes.clone())
                gt_boxes_xyxy[:, [0,2]] *= w0
                gt_boxes_xyxy[:, [1,3]] *= h0
                gt_cls = torch.zeros(len(gt), dtype=torch.int)
            else:
                gt_boxes_xyxy = torch.zeros((0, 4))
                gt_cls = torch.zeros(0, dtype=torch.int)

            pred = model(img_t)[0]
            dets = non_max_suppression(pred, conf_thres, iou_thres)[0]

            if dets is None or len(dets) == 0:
                if len(gt_cls) > 0:
                    # tp shape: (0, 1) — ap_per_class expects 2D
                    stats.append((np.zeros((0, 1), dtype=bool),
                                  np.zeros(0), np.zeros(0),
                                  gt_cls.numpy()))
                continue

            dets[:, :4] = scale_coords((imgsz, imgsz), dets[:, :4], (h0, w0))
            pred_boxes = dets[:, :4]
            pred_conf  = dets[:, 4]
            pred_cls   = dets[:, 5].int()

            # TP 판별 — shape (N, 1) for single IoU threshold
            correct = np.zeros((len(dets), 1), dtype=bool)
            if len(gt_boxes_xyxy) > 0:
                iou_mat = box_iou(pred_boxes, gt_boxes_xyxy.to(device))
                matched_gt = set()
                for pi in range(len(dets)):
                    best_iou, best_j = iou_mat[pi].max(0)
                    if best_iou >= 0.5 and best_j.item() not in matched_gt:
                        correct[pi, 0] = True
                        matched_gt.add(best_j.item())

            stats.append((correct, pred_conf.cpu().numpy(),
                          pred_cls.cpu().numpy(), gt_cls.numpy()))

    if not stats:
        return {'mAP@0.5': 0.0, 'Precision': 0.0, 'Recall': 0.0, 'F1': 0.0}

    tp_all  = np.concatenate([s[0] for s in stats], axis=0)
    conf_all = np.concatenate([s[1] for s in stats], axis=0)
    pcls_all = np.concatenate([s[2] for s in stats], axis=0)
    tcls_all = np.concatenate([s[3] for s in stats], axis=0)

    if tp_all.shape[0] > 0 and tp_all.any():
        p, r, ap, f1, _ = ap_per_class(tp_all, conf_all, pcls_all, tcls_all)
        mp, mr, map50, mf1 = p.mean(), r.mean(), ap.mean(), f1.mean()
    else:
        mp = mr = map50 = mf1 = 0.0

    return {'mAP@0.5': float(map50), 'Precision': float(mp),
            'Recall': float(mr), 'F1': float(mf1)}

print('평가 함수 정의 완료')

In [ ]:
# last400.pt 테스트셋 평가
results = evaluate(model, test_imgs)
print('\n=== last400.pt 베이스라인 결과 (Test set) ===')
for k, v in results.items():
    print(f'  {k:12s}: {v:.4f}')

## 6. 학습 데이터 수에 따른 성능 비교 (6개 가중치)

In [ ]:
import pandas as pd

weight_files = ['last15.pt', 'last50.pt', 'last100.pt',
                'last200.pt', 'last300.pt', 'last400.pt']
train_sizes  = [15, 50, 100, 200, 300, 400]

rows = []
for wf, sz in zip(weight_files, train_sizes):
    print(f'{wf} 평가 중...', end=' ', flush=True)
    m = load_model(wf)
    res = evaluate(m, test_imgs)
    res['학습 이미지 수'] = sz
    rows.append(res)
    print(f"mAP={res['mAP@0.5']:.4f}  Recall={res['Recall']:.4f}")
    del m
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

df = pd.DataFrame(rows)[['학습 이미지 수','mAP@0.5','Precision','Recall','F1']]
df = df.set_index('학습 이미지 수')
print('\n=== 학습 데이터 수별 성능 비교 ===')
print(df.round(4).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for col, color in [('mAP@0.5','steelblue'), ('Recall','tomato'), ('Precision','mediumseagreen'), ('F1','orange')]:
    ax.plot(df.index, df[col], marker='o', label=col, color=color, linewidth=2)
ax.set_xlabel('학습 이미지 수', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('학습 데이터 양에 따른 성능 변화', fontsize=14)
ax.legend(fontsize=11)
ax.set_xticks(train_sizes)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(os.path.join(BASELINE_DIR, 'performance_by_trainsize.png'), dpi=120)
plt.show()

## 7. 탐지 결과 시각화 (Test 이미지 샘플)

In [ ]:
import cv2

# best 모델 재로드
model = load_model('last400.pt')

def visualize_detections(img_path, conf_thres=0.25, iou_thres=0.5, imgsz=416):
    img_t, img0, ratio, pad = preprocess(img_path, imgsz)
    h0, w0 = img0.shape[:2]

    gt = load_labels(img_path)

    model.eval()
    with torch.no_grad():
        pred = model(img_t)[0]
    dets = non_max_suppression(pred, conf_thres, iou_thres)[0]

    img_rgb = cv2.cvtColor(img0, cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(img_rgb)

    # GT 박스 (초록)
    for box in gt:
        _, cx, cy, bw, bh = box
        x1 = (cx - bw/2) * w0;  y1 = (cy - bh/2) * h0
        rect = patches.Rectangle((x1, y1), bw*w0, bh*h0,
                                   linewidth=2, edgecolor='lime', facecolor='none')
        ax.add_patch(rect)

    # 예측 박스 (빨강)
    n_tp = 0
    if dets is not None and len(dets) > 0:
        dets[:, :4] = scale_coords((imgsz, imgsz), dets[:, :4], (h0, w0))
        for *xyxy, conf, cls in dets.tolist():
            x1, y1, x2, y2 = xyxy
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                       linewidth=1.5, edgecolor='red', facecolor='none',
                                       linestyle='--')
            ax.add_patch(rect)
            ax.text(x1, y1-3, f'{conf:.2f}', color='red', fontsize=8)
            n_tp += 1

    fname = os.path.basename(img_path)
    ax.set_title(f'{fname}\nGT: {len(gt)}개 (초록)  |  예측: {n_tp}개 (빨강)', fontsize=10)
    ax.axis('off')
    return fig

sample_imgs = test_imgs[:6]
fig_all, axes_all = plt.subplots(2, 3, figsize=(18, 10))

for i, img_path in enumerate(sample_imgs):
    img_t, img0, ratio, pad = preprocess(img_path)
    h0, w0 = img0.shape[:2]
    gt = load_labels(img_path)

    model.eval()
    with torch.no_grad():
        pred = model(img_t)[0]
    dets = non_max_suppression(pred, 0.25, 0.5)[0]

    img_rgb = cv2.cvtColor(img0, cv2.COLOR_BGR2RGB)
    ax = axes_all[i//3][i%3]
    ax.imshow(img_rgb)

    for box in gt:
        _, cx, cy, bw, bh = box
        x1 = (cx - bw/2)*w0; y1 = (cy - bh/2)*h0
        ax.add_patch(patches.Rectangle((x1,y1), bw*w0, bh*h0,
                     linewidth=2, edgecolor='lime', facecolor='none'))

    n_pred = 0
    if dets is not None and len(dets) > 0:
        dets[:, :4] = scale_coords((IMGSZ, IMGSZ), dets[:, :4], (h0, w0))
        for *xyxy, conf, cls in dets.tolist():
            x1, y1, x2, y2 = xyxy
            ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1,
                         linewidth=1.5, edgecolor='red', facecolor='none', linestyle='--'))
            ax.text(x1, y1-3, f'{conf:.2f}', color='red', fontsize=7)
            n_pred += 1

    ax.set_title(f'GT:{len(gt)} | 예측:{n_pred}', fontsize=9)
    ax.axis('off')

plt.suptitle('탐지 결과  초록=GT / 빨강=예측', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(BASELINE_DIR, 'detection_samples.png'), dpi=120, bbox_inches='tight')
plt.show()
print('시각화 저장 완료')

## 8. Confidence 분포 분석

In [ ]:
tp_confs = []
fp_confs = []
fn_count = 0

model.eval()
with torch.no_grad():
    for img_path in test_imgs:
        img_t, img0, ratio, pad = preprocess(img_path)
        h0, w0 = img0.shape[:2]
        gt = load_labels(img_path)

        pred = model(img_t)[0]
        dets = non_max_suppression(pred, 0.001, 0.5)[0]

        if gt.shape[0] > 0:
            gt_boxes = torch.tensor(gt[:, 1:], dtype=torch.float32)
            gt_xyxy = xywh2xyxy(gt_boxes.clone())
            gt_xyxy[:, [0,2]] *= w0
            gt_xyxy[:, [1,3]] *= h0

        if dets is None or len(dets) == 0:
            fn_count += len(gt)
            continue

        dets[:, :4] = scale_coords((IMGSZ, IMGSZ), dets[:, :4], (h0, w0))

        if gt.shape[0] > 0:
            iou_mat = box_iou(dets[:, :4], gt_xyxy.to(device))
            matched_gt = set()
            for pi in range(len(dets)):
                best_iou, best_j = iou_mat[pi].max(0)
                conf = dets[pi, 4].item()
                if best_iou >= 0.5 and best_j.item() not in matched_gt:
                    tp_confs.append(conf)
                    matched_gt.add(best_j.item())
                else:
                    fp_confs.append(conf)
            fn_count += len(gt) - len(matched_gt)
        else:
            fp_confs.extend(dets[:, 4].cpu().tolist())

fig, ax = plt.subplots(figsize=(9, 4))
bins = np.linspace(0, 1, 30)
if tp_confs:
    ax.hist(tp_confs, bins=bins, alpha=0.6, color='steelblue', label=f'TP ({len(tp_confs)}개)')
if fp_confs:
    ax.hist(fp_confs, bins=bins, alpha=0.6, color='tomato', label=f'FP ({len(fp_confs)}개)')
ax.set_xlabel('Confidence Score', fontsize=12)
ax.set_ylabel('개수')
ax.set_title(f'TP/FP Confidence 분포  |  FN 수: {fn_count}', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(BASELINE_DIR, 'confidence_dist.png'), dpi=120)
plt.show()

## 9. 결과 요약

In [ ]:
print('='*50)
print('베이스라인 결과 요약')
print('='*50)
print(f'테스트 이미지 수: {len(test_imgs)}')
print()
print('학습 데이터 수별 성능 (mAP@0.5):')
print(df[['mAP@0.5','Recall','Precision','F1']].round(4).to_string())
print()
best_row = df['mAP@0.5'].idxmax()
print(f'최고 성능: {best_row}장 학습 — mAP@0.5={df.loc[best_row,"mAP@0.5"]:.4f}')
print(f'TP: {len(tp_confs)}  FP: {len(fp_confs)}  FN: {fn_count}')